# Imports

In [ ]:
import os
import pandas as pd
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth import FastLanguageModel, FastModel
from transformers import DataCollatorForSeq2Seq, TextStreamer
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only, get_chat_template

# Useful Methods

In [ ]:
def load_unsloth_model(model_id="unsloth/gemma-3-4B-it", seq_len=2048, load_in_4bit=True, full_FT=False):
    model, tokenizer = FastModel.from_pretrained(
        model_name = model_id,
        max_seq_length = seq_len,
        load_in_4bit = load_in_4bit,
        load_in_8bit = not load_in_4bit,
        full_finetuning = full_FT,
        # token = "hf_...", # use one if using gated models
    )
    return model, tokenizer

# Useful Variables

In [ ]:
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

CHOSEN_MODELS = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct",
    "unsloth/Qwen2.5-3B",
    "unsloth/Qwen2.5-3B-bnb-4bit",
    "unsloth/Qwen2.5-3B-unsloth-bnb-4bit",
    "unsloth/Qwen2.5-14B",
    "unsloth/Qwen2.5-14B-unsloth-bnb-4bit",
]

In [ ]:
FILE_DIR = os.getcwd()
ROOT_DIR = os.path.dirname(os.path.dirname(FILE_DIR))
MODELS_DIR = os.path.join(ROOT_DIR, 'ft_output')
DATASET_DIR = os.path.join(ROOT_DIR, 'GFSlowFastSign/dataset/PHOENIX-2014-T/annotations/manual')
TRAIN_CSV = os.path.join(DATASET_DIR, 'PHOENIX-2014-T.train.corpus.csv')

In [ ]:
lang = "German" # "English"

SYS_PROMPT = """
    You are a sign language expert. Your task is to translate the sign language gloss for the given sentence.
    You will be given a sentence in {language} and you need to translate it into sign language gloss.
""".format(language=lang)

USER_PROMPT = """
    Here is the sentence:
    {sentence}
    Please translate it into sign language gloss.
"""

OUTPUT_PROMPT = """
    The gloss is: {gloss}
"""

# Load Dataset

In [ ]:
df = pd.read_csv(TRAIN_CSV, sep="|", header=0)
print(f'Read {len(df)} rows from {TRAIN_CSV}')
df = df.rename(columns={"orth": "Gloss", "translation": "Translation"})
df.head()

We need to extract only the needed columns `Gloss` and `Translation`.

In [ ]:
# Extract only Gloss and Translation
data_df = df[["Gloss", "Translation"]]
data_df.head()

# Load LLM
Remember that we can pick any LLM from the ones below.

In [ ]:
print(f'Available models:')
for model in CHOSEN_MODELS:
    print(f' - {model}')

In [ ]:
max_seq_length = 2048
# max_seq_length = 4096
chosen_model = CHOSEN_MODELS[3]
print(f'Using model: {chosen_model}')

In [ ]:
model, tokenizer = load_unsloth_model(chosen_model, seq_len=2048, load_in_4bit=False, full_FT=False)

In [ ]:
# Do model patching and add fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, # Suggested to be equal to the rank r, or double it.
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

## Preprocess Data

We further process the data so that the dataframe contains only `questions` and `answers`.

In your case `questions` correspond to users asking the LLM to gloss the input sentence, and `answers` are translations of the sentences.

In [ ]:
data_df = data_df.rename(columns={"Gloss": "Output", "Translation": "User"})

data_df["User"] = data_df["User"].apply(
    lambda x: USER_PROMPT.format(sentence=x).strip()
)

data_df["Output"] = data_df["Output"].apply(
    lambda x: OUTPUT_PROMPT.format(gloss=x).strip()
)

data_df["System"] = SYS_PROMPT

data_df.head()

Let's reorder the columns for better readability.

In [ ]:
data_df = data_df[["System", "User", "Output"]]
data_df.head()

In [ ]:
converted = []
for _, row in data_df.iterrows():
    messages = [
        {"from": "system", "value": row["System"].strip()},
        {"from": "human", "value": row["User"].strip()},
        {"from": "gpt", "value": row["Output"].strip()},
    ]
    converted.append({"conversations": messages})

print(f'Converted {len(converted)} rows to chat format.')

We are now ready to load the data with in a `HuggingFace` dataset.

In [ ]:
data = Dataset.from_list(converted)

Let's set up the training dataset using the `standardize_sharegpt` method. More info on this method [here](https://github.com/unslothai/unsloth/blob/6c234d5a66adb76b9b93fb0f2445648199d88e66/unsloth/chat_templates.py#L43).

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

In [ ]:
data

In [ ]:
data = standardize_sharegpt(data)
data = data.map(formatting_prompts_func, batched=True)
data

We retrieve only 1000 instances from the total amount just to test the training.

In [ ]:
# train_data = data.shuffle(seed=42).select(range(3500))
train_data = data.shuffle(seed=42)
train_data

Here an example of the final data instance.

In [ ]:
train_data[0]

## Let's Fine Tune the LLM

In [ ]:
ft_config = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 4,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60, # Comment this to set num_train_epochs (longer training)
        num_train_epochs = 3, # 3 Suggested as best number of epochs https://docs.unsloth.ai/basics/tutorial-how-to-finetune-llama-3-and-use-in-ollama#id-11.-inference-running-the-model
        learning_rate=2e-4, # Suggested values: 2e-4, 1e-4, 5e-5, 2e-5
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim="adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/home/ivan/GFSlowFastSign/ft_output", # "outputs",
    )

In [ ]:
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_data,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    args = ft_config
)

In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
trainer.train()

Let's save the model.

In [ ]:
model.save_pretrained("/home/ivan/GFSlowFastSign/ft_output/gloss_lama")
tokenizer.save_pretrained("/home/ivan/GFSlowFastSign/ft_output/gloss_lama")

# Let's test the tuned model

In [ ]:
def prepare_msg(tokenizer, msg=None):
    
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "llama-3.1",
    )

    messages = [
        {"role": "user", "content": f"{msg}"},
    ]
    
    return tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Must add for generation
        return_tensors = "pt",
    ).to("cuda")
    

def gen_response(input_ids, tuned_model, max_new_tokens=128, temperature=0.7, top_k=50, top_p=0.95,
                 text_streamer=None, pad_token_id=None):
    _ = tuned_model.generate(
        input_ids,
        max_new_tokens = max_new_tokens,
        do_sample = True,
        temperature = temperature, 
        top_k = top_k,   
        top_p = top_p, 
        pad_token_id = pad_token_id,
        streamer=text_streamer,
    )
    

def standard_gen(model_to_use, input_ids=None, max_new_tokens=128, temperature=0.7, min_p=0.1, use_cache=True, text_streamer=None):
    _ = model_to_use.generate(input_ids = input_ids, max_new_tokens = max_new_tokens, use_cache = use_cache,
                         temperature = temperature, min_p = min_p, streamer=text_streamer,)

In [ ]:
sen = data_df.iloc[5020, 1]
print(f'User prompt:\n{sen}')

Let's load the model for FastInference, the TextStreamer (needed to stream the model's output during generation) and finally we also prepare the model's input.

In [ ]:
model = FastLanguageModel.for_inference(model)

inputs = prepare_msg(tokenizer, sen)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

In [ ]:
standard_gen(model, input_ids=inputs, max_new_tokens=64, temperature=1.5, min_p=0.1, use_cache=True, text_streamer=text_streamer)

In [ ]:
standard_gen(model, input_ids=inputs, max_new_tokens=128, temperature=0.7, min_p=0.1, use_cache=True)

In [ ]:
gen_response(inputs, model, max_new_tokens=128, temperature=0.1, top_k=50, top_p=0.95,
             text_streamer=text_streamer, pad_token_id=tokenizer.eos_token_id)

In [ ]:
gen_response(inputs, model, max_new_tokens=128, temperature=0.7, top_k=50, top_p=0.95,
             text_streamer=text_streamer, pad_token_id=tokenizer.eos_token_id)

In [ ]:
gen_response(inputs, model, max_new_tokens=128, temperature=0.5, top_k=50, top_p=0.95,
             text_streamer=text_streamer, pad_token_id=tokenizer.eos_token_id)